In [ ]:
import os
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# 1. Load and clean
df = pd.read_csv("supply_chain_dataset1.csv")
target = "Stockout_Flag"

df = df.drop_duplicates()
df = df.dropna(subset=[target]).reset_index(drop=True)

# Exclude target and potentially unavailable future forecast
X = df.drop(
    columns=[target, "Demand_Forecast"],
    errors="ignore"
)
y = df[target]

# Convert date into calendar features
if "Date" in X.columns:
    dates = pd.to_datetime(X.pop("Date"), errors="coerce")
    X["date_year"] = dates.dt.year
    X["date_month"] = dates.dt.month
    X["date_day"] = dates.dt.day
    X["date_dayofweek"] = dates.dt.dayofweek

X = X.select_dtypes(exclude=["complex"])
X = X.dropna(axis=1, how="all")

# 2. Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 3. Identify feature types
numeric_cols = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_cols = [
    col for col in X_train.columns
    if col not in numeric_cols
]

# 4. Preprocessing
preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_cols),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols)
])

# 5. Build model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# 6. Train and evaluate
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred, zero_division=0))

if y.nunique() == 2:
    print("ROC-AUC:", round(
        roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]), 4
    ))

# 7. Save
os.makedirs("saved_models", exist_ok=True)

joblib.dump(model, "saved_models/supply_chain_model.pkl")
joblib.dump({
    "target": target,
    "features": X.columns.tolist(),
    "numeric_features": numeric_cols,
    "categorical_features": categorical_cols
}, "saved_models/supply_chain_model_metadata.pkl")

print("Supply chain model and metadata saved.")

Accuracy: 1.0
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00      2995

    accuracy                           1.00      2995
   macro avg       1.00      1.00      1.00      2995
weighted avg       1.00      1.00      1.00      2995

Supply chain model and metadata saved.
